In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

In [2]:
# Fetch dataset from UCI
energy_efficiency = fetch_ucirepo(id=242)

# Separate features and targets
X = energy_efficiency.data.features.values
y = energy_efficiency.data.targets['Y1'].values

# Check shape
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (768, 8)
y shape: (768,)


In [3]:
# Manually split data into 80% train and 20% test
np.random.seed(42)

# Generate random indices
indices = np.random.permutation(768)

# 80/20 split
split = int(0.8 * 768)
train_idx = indices[:split]
test_idx = indices[split:]

# Split X and y
X_train = X[train_idx]
X_test = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 614
Testing samples: 154


In [4]:
# Standardize features from scratch (mean=0, std=1)
# Formula: x_scaled = (x - mean) / std

# Calculate mean and std from training data only
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)

# Apply scaling to both train and test
X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std

print("Feature means:", X_mean.round(3))
print("Feature stds:", X_std.round(3))

Feature means: [7.65000e-01 6.71356e+02 3.19537e+02 1.75909e+02 5.28400e+00 3.50300e+00
 2.36000e-01 2.82100e+00]
Feature stds: [ 0.106 88.179 43.289 44.978  1.75   1.121  0.133  1.541]


In [5]:
# Linear Regression from scratch using the Normal Equation
# Formula: β = (XᵀX)⁻¹Xᵀy

# Add column of ones to X_train for the intercept term
X_b = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]

# Normal equation
beta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y_train

# Separate intercept and coefficients
intercept = beta[0]
coefficients = beta[1:]

print("Intercept:", round(intercept, 3))
print("Coefficients:", coefficients.round(3))

Intercept: 22.656
Coefficients: [ -8.449 -11.048   3.59    1.321   7.289  -0.086   2.593   0.305]


In [6]:
# Linear Regression from scratch using the Normal Equation
# Formula: β = (XᵀX)⁻¹Xᵀy

# Add column of ones to X_train for the intercept term
X_b = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]

# Normal equation
beta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y_train

# Separate intercept and coefficients
intercept = beta[0]
coefficients = beta[1:]

print("Intercept:", round(intercept, 3))
print("Coefficients:", coefficients.round(3))

Intercept: 22.656
Coefficients: [ -8.449 -11.048   3.59    1.321   7.289  -0.086   2.593   0.305]


In [7]:
# Add column of ones to X_test for intercept
X_test_b = np.c_[np.ones((X_test_scaled.shape[0], 1)), X_test_scaled]

# Make predictions using our learned beta
y_pred = X_test_b @ beta

# Calculate accuracy metrics from scratch
# R² score
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2 = 1 - (ss_res / ss_tot)

# MAE
mae = np.mean(np.abs(y_test - y_pred))

# RMSE
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))

print("Linear Regression:")
print(f"  R²:   {r2:.3f}")
print(f"  MAE:  {mae:.3f}")
print(f"  RMSE: {rmse:.3f}")

Linear Regression:
  R²:   0.901
  MAE:  2.089
  RMSE: 2.951


In [8]:
# Ridge Regression from scratch
# Formula: β = (XᵀX + λI)⁻¹Xᵀy
# Only difference from Linear Regression is the λI term added to XᵀX

# Set lambda (regularization strength)
lambda_ = 1.0

# Add column of ones to X_train for intercept
X_b_train = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]

# Create identity matrix
I = np.eye(X_b_train.shape[1])

# Don't regularize the intercept term
I[0, 0] = 0

# Ridge Normal Equation
beta_ridge = np.linalg.inv(X_b_train.T @ X_b_train + lambda_ * I) @ X_b_train.T @ y_train

# Make predictions
y_pred_ridge = X_test_b @ beta_ridge

# Calculate metrics
ss_res_ridge = np.sum((y_test - y_pred_ridge) ** 2)
r2_ridge = 1 - (ss_res_ridge / ss_tot)
mae_ridge = np.mean(np.abs(y_test - y_pred_ridge))
rmse_ridge = np.sqrt(np.mean((y_test - y_pred_ridge) ** 2))

print("Ridge Regression:")
print(f"  Lambda: {lambda_}")
print(f"  R²:   {r2_ridge:.3f}")
print(f"  MAE:  {mae_ridge:.3f}")
print(f"  RMSE: {rmse_ridge:.3f}")
print(f"  Coefficients: {beta_ridge[1:].round(3)}")

Ridge Regression:
  Lambda: 1.0
  R²:   0.908
  MAE:  2.009
  RMSE: 2.841
  Coefficients: [-5.998 -3.162  0.86  -3.513  7.62  -0.084  2.586  0.306]


In [9]:
# Lasso Regression from scratch using Coordinate Descent
# Uses soft thresholding to zero out small coefficients

def soft_threshold(rho, lambda_):
    # This is the core of Lasso
    if rho > lambda_:
        return rho - lambda_
    elif rho < -lambda_:
        return rho + lambda_
    else:
        return 0.0

def lasso_coordinate_descent(X, y, lambda_=1.0, max_iter=1000, tol=1e-4):
    # Add intercept column
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    n, p = X_b.shape
    
    # Initialize coefficients to zero
    beta = np.zeros(p)
    
    for iteration in range(max_iter):
        beta_old = beta.copy()
        
        # Update each coefficient one at a time
        for j in range(p):
            # Calculate residual excluding current coefficient
            residual = y - X_b @ beta + X_b[:, j] * beta[j]
            
            # Calculate rho
            rho = X_b[:, j] @ residual / n
            
            # Don't penalize intercept
            if j == 0:
                beta[j] = rho
            else:
                beta[j] = soft_threshold(rho, lambda_ / n)
        
        # Check convergence
        if np.max(np.abs(beta - beta_old)) < tol:
            print(f"Converged at iteration {iteration + 1}")
            break
    
    return beta

# Train Lasso
lambda_ = 0.1
beta_lasso = lasso_coordinate_descent(
    X_train_scaled, 
    y_train, 
    lambda_=lambda_
)

# Make predictions
X_test_b = np.c_[np.ones((X_test_scaled.shape[0], 1)), X_test_scaled]
y_pred_lasso = X_test_b @ beta_lasso

# Calculate metrics
ss_res_lasso = np.sum((y_test - y_pred_lasso) ** 2)
r2_lasso = 1 - (ss_res_lasso / ss_tot)
mae_lasso = np.mean(np.abs(y_test - y_pred_lasso))
rmse_lasso = np.sqrt(np.mean((y_test - y_pred_lasso) ** 2))

print("Lasso Regression:")
print(f"  Lambda: {lambda_}")
print(f"  R²:   {r2_lasso:.3f}")
print(f"  MAE:  {mae_lasso:.3f}")
print(f"  RMSE: {rmse_lasso:.3f}")
print(f"  Coefficients: {beta_lasso[1:].round(3)}")
print(f"  Zeroed out features: {np.sum(beta_lasso[1:] == 0)}")

Converged at iteration 790
Lasso Regression:
  Lambda: 0.1
  R²:   0.908
  MAE:  2.021
  RMSE: 2.850
  Coefficients: [-7.27  -7.638  2.423 -0.544  7.304 -0.086  2.593  0.305]
  Zeroed out features: 0


In [11]:
"""
plots.py  —  Generate all figures for the energy regression paper.

Run this script from the same directory as your Jupyter notebook.
It will save four PDF files that can be included via \\includegraphics in the .tex file:

    corr_heatmap.pdf      — Pearson correlation matrix (Section 2.4)
    pred_vs_actual.pdf    — Predicted vs. actual scatter (Section 4.5)
    residuals.pdf         — Residuals vs. fitted values  (Section 4.5)
    coef_comparison.pdf   — Coefficient bar chart        (Section 4.5)

Requirements:  numpy, matplotlib, seaborn, ucimlrepo
    pip install ucimlrepo seaborn matplotlib numpy
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from ucimlrepo import fetch_ucirepo

# ── consistent style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "serif",
    "font.size":        11,
    "axes.titlesize":   12,
    "axes.labelsize":   11,
    "legend.fontsize":  10,
    "figure.dpi":       150,
    "savefig.dpi":      300,
    "savefig.bbox":     "tight",
    "savefig.format":   "pdf",
})

FEAT_LABELS = ["$X_1$\nCompactness", "$X_2$\nSurface A.",
               "$X_3$\nWall A.",    "$X_4$\nRoof A.",
               "$X_5$\nHeight",     "$X_6$\nOrientation",
               "$X_7$\nGlazing A.", "$X_8$\nGlazing Dist.",
               "$Y_1$\nHeat Load"]

FEAT_SHORT  = ["$X_1$","$X_2$","$X_3$","$X_4$",
               "$X_5$","$X_6$","$X_7$","$X_8$"]

# ── 1. load + split + scale  (identical to notebook) ─────────────────────────
print("Fetching dataset …")
ds = fetch_ucirepo(id=242)
X  = ds.data.features.values          # (768, 8)
y  = ds.data.targets["Y1"].values     # (768,)

np.random.seed(42)
idx   = np.random.permutation(768)
split = int(0.8 * 768)
tr, te = idx[:split], idx[split:]

X_train, X_test = X[tr], X[te]
y_train, y_test = y[tr], y[te]

mu, sigma       = X_train.mean(0), X_train.std(0)
Xtr = (X_train - mu) / sigma
Xte = (X_test  - mu) / sigma

# ── 2. OLS ───────────────────────────────────────────────────────────────────
Xb     = np.c_[np.ones(Xtr.shape[0]), Xtr]
beta   = np.linalg.inv(Xb.T @ Xb) @ Xb.T @ y_train

Xb_te  = np.c_[np.ones(Xte.shape[0]), Xte]
y_ols  = Xb_te @ beta

# ── 3. Ridge ─────────────────────────────────────────────────────────────────
lam_r  = 1.0
I_mod  = np.eye(Xb.shape[1]); I_mod[0,0] = 0
beta_r = np.linalg.inv(Xb.T @ Xb + lam_r * I_mod) @ Xb.T @ y_train
y_rdg  = Xb_te @ beta_r

# ── 4. Lasso ─────────────────────────────────────────────────────────────────
def soft(rho, lam):
    return np.sign(rho) * max(abs(rho) - lam, 0.0)

def lasso_cd(X, y, lam=0.1, max_iter=1000, tol=1e-4):
    Xb = np.c_[np.ones(X.shape[0]), X]
    n, p = Xb.shape
    b = np.zeros(p)
    for _ in range(max_iter):
        b_old = b.copy()
        for j in range(p):
            r   = y - Xb @ b + Xb[:, j] * b[j]
            rho = Xb[:, j] @ r / n
            b[j] = rho if j == 0 else soft(rho, lam / n)
        if np.max(np.abs(b - b_old)) < tol:
            break
    return b

beta_l = lasso_cd(Xtr, y_train, lam=0.1)
y_las  = Xb_te @ beta_l

coefs_ols  = beta[1:]
coefs_rdg  = beta_r[1:]
coefs_las  = beta_l[1:]


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Correlation heatmap
# ════════════════════════════════════════════════════════════════════════════
Xy    = np.c_[X, y]
corr  = np.corrcoef(Xy.T)

fig, ax = plt.subplots(figsize=(7, 6))
mask    = np.zeros_like(corr, dtype=bool)           # show full matrix
im      = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.82, label="Pearson $r$")

ax.set_xticks(range(9)); ax.set_xticklabels(FEAT_LABELS, fontsize=8)
ax.set_yticks(range(9)); ax.set_yticklabels(FEAT_LABELS, fontsize=8)
ax.set_title("Pearson Correlation Matrix — Features and Heating Load $Y_1$")

# annotate cells
for i in range(9):
    for j in range(9):
        val = corr[i, j]
        col = "white" if abs(val) > 0.6 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=7, color=col)

plt.tight_layout()
plt.savefig("corr_heatmap.pdf")
print("Saved  corr_heatmap.pdf")
plt.close()


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Predicted vs. Actual (all three models, side by side)
# ════════════════════════════════════════════════════════════════════════════
models  = [("OLS",   y_ols,  "#1f77b4"),
           ("Ridge ($\\lambda=1.0$)", y_rdg, "#ff7f0e"),
           ("Lasso ($\\lambda=0.1$)", y_las, "#2ca02c")]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
lo, hi = 5, 45
for ax, (name, yp, col) in zip(axes, models):
    ax.scatter(y_test, yp, s=18, alpha=0.55, color=col, edgecolors="none")
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="$\\hat{y}=y$")
    ss_r = np.sum((y_test - yp)**2)
    ss_t = np.sum((y_test - y_test.mean())**2)
    r2   = 1 - ss_r / ss_t
    mae  = np.mean(np.abs(y_test - yp))
    rmse = np.sqrt(np.mean((y_test - yp)**2))
    ax.set_title(name)
    ax.set_xlabel("Actual heating load (kWh/m$^2$)")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.legend(loc="upper left", fontsize=8)
    ax.text(0.04, 0.95, f"$R^2={r2:.3f}$\nMAE $={mae:.3f}$\nRMSE $={rmse:.3f}$",
            transform=ax.transAxes, va="top", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7))

axes[0].set_ylabel("Predicted heating load (kWh/m$^2$)")
fig.suptitle(r"Predicted vs.\ Actual Heating Load — Test Set ($n=154$)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("pred_vs_actual.pdf")
print("Saved  pred_vs_actual.pdf")
plt.close()


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Residuals vs. Fitted (OLS only, with Ridge & Lasso overlay)
# ════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, (name, yp, col) in zip(axes, models):
    resid = y_test - yp
    ax.scatter(yp, resid, s=18, alpha=0.55, color=col, edgecolors="none")
    ax.axhline(0, color="k", lw=1, ls="--")
    ax.set_title(name)
    ax.set_xlabel("Fitted value (kWh/m$^2$)")
    ax.set_xlim(5, 45)

axes[0].set_ylabel("Residual $y - \\hat{y}$ (kWh/m$^2$)")
fig.suptitle(r"Residuals vs.\ Fitted Values — Test Set", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("residuals.pdf")
print("Saved  residuals.pdf")
plt.close()


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Coefficient comparison bar chart
# ════════════════════════════════════════════════════════════════════════════
x      = np.arange(len(FEAT_SHORT))
width  = 0.26

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width, coefs_ols, width, label="OLS",             color="#1f77b4", alpha=0.85)
b2 = ax.bar(x,         coefs_rdg, width, label="Ridge ($\\lambda=1.0$)", color="#ff7f0e", alpha=0.85)
b3 = ax.bar(x + width, coefs_las, width, label="Lasso ($\\lambda=0.1$)", color="#2ca02c", alpha=0.85)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(FEAT_SHORT, fontsize=11)
ax.set_ylabel("Coefficient (standardised scale)")
ax.set_title("Fitted Coefficients — OLS vs. Ridge vs. Lasso\n"
             "(Intercept $\\beta_0 = 22.307$ omitted for all models)")
ax.legend()

# annotate the two Lasso zeros
for i, c in enumerate(coefs_las):
    if c == 0.0:
        ax.annotate("0", xy=(x[i] + width, 0.25), ha="center",
                    fontsize=8, color="#2ca02c", fontweight="bold")

plt.tight_layout()
plt.savefig("coef_comparison.pdf")
print("Saved  coef_comparison.pdf")
plt.close()

print("\nAll figures saved.  Copy the four PDFs next to your .tex file and")
print("replace the \\fbox placeholders with \\includegraphics{filename}.")

Fetching dataset …
Saved  corr_heatmap.pdf
Saved  pred_vs_actual.pdf
Saved  residuals.pdf
Saved  coef_comparison.pdf

All figures saved.  Copy the four PDFs next to your .tex file and
replace the \fbox placeholders with \includegraphics{filename}.
